# Steering Vectors

Extract a direction in activation space that represents a concept (e.g. positive vs negative sentiment),
then add it during inference to steer the model's output.

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Extract a steering vector

The steering vector is the difference in activations between a positive and negative example at a chosen layer.

In [ ]:
vector = model.steer_vector(
    positive="Love",
    negative="Hate",
    at="transformer.h.8",
)
print(f"Vector shape: {vector.shape}")
print(f"Vector norm:  {vector.float().norm():.2f}")

## Apply the steering vector

Add the vector (scaled) to the model's activations during inference.
Compare the top predictions with and without steering.

In [ ]:
result = model.steer(
    "The weather today is",
    vector=vector,
    at="transformer.h.8",
    scale=3.0,
)

## Try different scales

Higher scales push the model harder in the steering direction.

In [ ]:
for scale in [1.0, 3.0, 5.0, 10.0]:
    print(f"\n{'='*50}")
    print(f"Scale: {scale}")
    model.steer(
        "The movie was",
        vector=vector,
        at="transformer.h.8",
        scale=scale,
    )

## Try different layers

Steering at earlier layers has a different effect than later layers.

In [ ]:
for layer_idx in [0, 4, 8, 11]:
    layer_name = f"transformer.h.{layer_idx}"
    vec = model.steer_vector("Love", "Hate", at=layer_name)
    print(f"\n{'='*50}")
    print(f"Steering at layer {layer_idx}")
    model.steer(
        "The weather today is",
        vector=vec,
        at=layer_name,
        scale=3.0,
    )

## Export comparison chart

In [ ]:
model.steer(
    "The weather today is",
    vector=vector,
    at="transformer.h.8",
    scale=3.0,
    save="steering_comparison.png",
)